# Laboratorio 4 (Parte 2) — Inciso 2: Construcción de la variable respuesta

A partir de `clorofila` (clorofila-a en µg/L, calculada en el inciso 1 mediante el polinomio NDCI de Mishra & Mishra) se construye la variable respuesta binaria `alta_cianobacteria`, se analiza su distribución global, por lago y por fecha, se cuantifica el desbalance de clases y se identifican las variables que deben excluirse como predictoras por fuga de información.

In [ ]:
import sys
from pathlib import Path

sys.path.append(str(Path.cwd().parent))

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

from src.config import LAGOS, RUTA_DATA_PROCESSED, RUTA_FIGURAS
from src.modelado import EXCLUIDAS, PREDICTORES, UMBRAL_CLOROFILA, construir_respuesta, distribucion_respuesta

colores = {"atitlan": "teal", "amatitlan": "darkorange"}

## 2.1 y 2.2 Construcción del umbral y justificación científica

Se define `alta_cianobacteria = 1` cuando `clorofila >= 10 µg/L`, y `0` en caso contrario.

In [ ]:
dataset = pd.read_parquet(RUTA_DATA_PROCESSED / "dataset_ml.parquet")
dataset = construir_respuesta(dataset)
print(f"Umbral utilizado: clorofila-a >= {UMBRAL_CLOROFILA} µg/L")
dataset[["lago", "fecha", "clorofila", "alta_cianobacteria"]].head()

**Justificación del punto de corte (10 µg/L de clorofila-a).**

- La Organización Mundial de la Salud (OMS/WHO, 2003, *Guidelines for Safe Recreational Water Environments*, Vol. 1) define el **Alert Level 1** para aguas recreativas en clorofila-a >= 10 µg/L (típicamente acompañada de cianobacterias >= 20,000 células/mL), nivel a partir del cual recomienda vigilancia activa por riesgo de irritación de piel y síntomas gastrointestinales leves. Es el umbral regulatorio más citado para "presencia alta" de cianobacterias en cuerpos de agua recreativos, y es el criterio principal usado aquí.
- Coincide con el límite inferior del estado **eutrófico** en los esquemas de clasificación trófica de lagos: la OECD (1982, *Eutrophication of Waters: Monitoring, Assessment and Control*) ubica el rango eutrófico en clorofila-a media de 8 a 25 µg/L, y el Índice de Estado Trófico de Carlson (1977, *A trophic state index for lakes*, TSI(Chl) = 9.81·ln(Chl) + 30.6) marca el límite mesotrófico-eutrófico (TSI = 50) en clorofila-a ≈ 7 µg/L.
- Los tres criterios (salud pública, estado trófico OECD, TSI de Carlson) convergen en un rango de 7 a 10 µg/L; se adopta el valor de la OMS, 10 µg/L, por ser el más directamente ligado a riesgo sanitario y el más usado en monitoreo operativo de floraciones, que es el uso previsto de este modelo.
- El umbral se aplica sobre `clorofila`, no sobre `ndci` directamente, porque `clorofila` ya está en unidades físicas (µg/L) comparables con la bibliografía citada, mientras que `ndci` es adimensional.

## 2.3 Distribución de la variable respuesta

In [ ]:
dist = distribucion_respuesta(dataset)
print("Global:")
print(dist["global"])
print()
print("Por lago:")
dist["por_lago"]

In [ ]:
por_lago_fecha = dist["por_lago_fecha"].reset_index()

fig, axes = plt.subplots(1, 2, figsize=(13, 5))

axes[0].bar(dist["por_lago"].index, dist["por_lago"]["proporcion_alta"] * 100,
            color=[colores[l] for l in dist["por_lago"].index])
axes[0].set_ylabel("% de observaciones con alta_cianobacteria = 1")
axes[0].set_title("Proporción de clase positiva por lago")
axes[0].set_xticklabels([LAGOS[l]["nombre"] for l in dist["por_lago"].index])
axes[0].grid(alpha=0.3, axis="y")

for lago in LAGOS:
    sub = por_lago_fecha[por_lago_fecha["lago"] == lago].sort_values("fecha")
    axes[1].plot(sub["fecha"], sub["proporcion_alta"] * 100, marker="o", label=LAGOS[lago]["nombre"], color=colores[lago])
axes[1].set_ylabel("% de observaciones con alta_cianobacteria = 1")
axes[1].set_title("Proporción de clase positiva por fecha")
axes[1].tick_params(axis="x", rotation=45)
axes[1].legend()
axes[1].grid(alpha=0.3)

fig.suptitle("Distribución de la variable respuesta")
fig.tight_layout()
fig.savefig(RUTA_FIGURAS / "p2_distribucion_respuesta.png", dpi=150)
plt.show()

Amatitlán tiene una proporción de clase positiva muy superior a Atitlán en todas las fechas, coherente con su mayor deterioro ambiental reportado en la Parte I. En Atitlán la proporción se mantiene baja y estable, con un repunte moderado en las fechas de 2026. En Amatitlán la proporción es alta desde el inicio de la serie y muestra picos pronunciados en 2026-04-28 y 2026-06-19, que coinciden con los eventos de floración ya identificados en la Parte I.

## 2.4 Desbalance de clases

In [ ]:
n0 = int((dataset["alta_cianobacteria"] == 0).sum())
n1 = int((dataset["alta_cianobacteria"] == 1).sum())
print(f"Clase 0 (ausencia/baja): {n0:,} ({n0 / len(dataset):.1%})")
print(f"Clase 1 (alta):          {n1:,} ({n1 / len(dataset):.1%})")
print(f"Razón clase 0 : clase 1  = {n0 / n1:.1f} : 1")

for lago in LAGOS:
    sub = dataset[dataset["lago"] == lago]
    n0_l, n1_l = int((sub["alta_cianobacteria"] == 0).sum()), int((sub["alta_cianobacteria"] == 1).sum())
    print(f"{lago}: razón 0:1 = {n0_l / max(n1_l, 1):.1f} : 1")

Existe un desbalance de clases considerable, dominado por Atitlán (más observaciones totales y menor proporción positiva). Consecuencias sobre el entrenamiento y la evaluación:

- **Entrenamiento**: un clasificador puede minimizar la función de pérdida global prediciendo casi siempre la clase mayoritaria (0), logrando *accuracy* alta sin aprender a distinguir la clase de interés. Se mitiga con `class_weight="balanced"` en Regresión Logística y Random Forest, y con la división estratificada del inciso 4.2, que preserva la proporción de clases en entrenamiento y prueba.
- **Evaluación**: el *accuracy* deja de ser informativo, ya que un modelo trivial que siempre predice 0 obtendría un *accuracy* cercano a la proporción de la clase mayoritaria. Por eso el inciso 5 reporta también *precision*, *recall*, *F1* y *ROC-AUC*, y prioriza el *recall* de la clase 1 dado el costo ambiental de no detectar una floración real (inciso 5.3).

## 2.5 Variables que no pueden utilizarse como predictoras

In [ ]:
print("Excluidas (participan en la construcción de la variable respuesta):")
print(EXCLUIDAS)
print()
print("Predictoras candidatas:")
print(PREDICTORES)

assert set(PREDICTORES).isdisjoint(EXCLUIDAS)
assert "alta_cianobacteria" not in PREDICTORES

`clorofila` es la variable a partir de la cual se construye directamente `alta_cianobacteria`: incluirla como predictora haría que el modelo aprenda literalmente el umbral, no un patrón espectral. `ndci` queda excluida porque `clorofila` es una transformación polinomial directa de `ndci` (correlación de Spearman de 1.00, reportada en el inciso 1). `rojo` (B04) y `b05` (B05) quedan excluidas porque `ndci = (B05 - B04) / (B05 + B04)` se calcula directamente a partir de ellas, de modo que también producirían fuga de información indirecta hacia la respuesta.

`fai` y `ndvi` sí se conservan como predictoras a pesar de que su fórmula también usa B04: a diferencia de `ndci`, no son la variable a partir de la cual se definió el umbral de la respuesta, combinan B04 con otras bandas (B07, B8A) y capturan una señal distinta (materia flotante y vegetación, respectivamente) que aporta información adicional sin ser una transformación directa de la respuesta.

### Self-check

In [ ]:
assert set(PREDICTORES).isdisjoint(EXCLUIDAS), "una predictora está marcada también como excluida"
assert dataset["alta_cianobacteria"].isin([0, 1]).all(), "la variable respuesta no es binaria"
assert (dataset["alta_cianobacteria"] == 1).sum() == (dataset["clorofila"] >= UMBRAL_CLOROFILA).sum()
assert n0 + n1 == len(dataset)
print("OK: variable respuesta binaria, consistente con el umbral, y sin traslape entre predictoras y excluidas.")